# 국민여행조사 3개년 변수 조화 통합

## 분석 목적과 범위

- 2023·2024·2025년 국내여행 SAV를
  `.claude/skills/use-national-travel-survey/references/variable-harmonization.md`의
  규칙과 사용자 검증 대응표
  (`2023_2025_국민여행조사_코드북_대응변수_및_처리방식.csv`)에 따라 의미
  기준으로 통합한다.
- 분석 단위는 응답자·조사회차이며 기본 키는 `YEAR + ID`다. 원본 SAV는
  읽기 전용으로 사용한다.
- 셀 순서: 데이터 로드 → 2023년 변수명 수정 → 새로 생성해야 하는 변수
  추가·코드값 수정 전처리 → 데이터 미리보기 → 코드북 작성 → 통합
  데이터와 코드북 저장.
- 참고한 인터넷 사이트(A5A_1~3)의 2023년 자체 값 5(여행 관련 블로그)는
  순위와 무관하게 통합 코드 9(기타)로 재코딩한다. 2024·2025년 코드도
  동일한 9개 범주 체계로 재코딩되므로 값 5는 어느 연도에서도 남지
  않는다(사용자 확인 완료).
- 비여행이유는 통합 열 이름을 2024·2025년 이름인 `ZQ1_1~3`으로
  유지하고, 값 체계는 2023년 코드(11=기타)로 통일한다(사용자 확인
  완료).

## 환경, 공통 경로와 재현성 설정

In [17]:
from pathlib import Path
import re
import sys


current_directory = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in (current_directory, *current_directory.parents)
        if (candidate / "src" / "path.py").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "프로젝트 루트를 찾을 수 없습니다. 프로젝트 내부에서 실행하세요."
    )
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json  # noqa: E402

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import pyreadstat  # noqa: E402

from src.path import (  # noqa: E402
    PREPROCESS_DATA_DIR,
    NATIONAL_TRAVEL_SURVEY_RAW_DATA_DIR,
    NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR,
)


YEARS = (2023, 2024, 2025)
SAV_FILES = {
    2023: "2023년_국민여행조사_국내여행.SAV",
    2024: "2024년_국민여행조사_국내여행.sav",
    2025: "2025년_국민여행조사_국내여행.SAV",
}
EXPECTED_ROWS = {2023: 52111, 2024: 51754, 2025: 52185}
REFERENCE_YEAR = 2025
OUTPUT_DATA_PATH = (
    NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR / "national_travel_survey_2023_2025.csv"
)
OUTPUT_CODEBOOK_PATH = (
    NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR / "national_travel_survey_2023_2025_codebook.csv"
)

## 데이터 로드

3개년 SAV에서 영문자로 시작하는 공식 원변수만 읽는다(값 라벨 형식은
적용하지 않고 원코드를 유지한다).

In [2]:
metadata_by_year = {}
raw_data_by_year = {}
official_variables_by_year = {}

for year in YEARS:
    sav_path = NATIONAL_TRAVEL_SURVEY_RAW_DATA_DIR / f"y{year}" / SAV_FILES[year]
    _, metadata_only = pyreadstat.read_sav(sav_path, metadataonly=True)
    official_variables_by_year[year] = [
        name
        for name in metadata_only.column_names
        if re.match(r"^[A-Za-z]", name)
    ]
    data, metadata = pyreadstat.read_sav(
        sav_path,
        usecols=official_variables_by_year[year],
        apply_value_formats=False,
    )
    assert len(data) == EXPECTED_ROWS[year]
    assert data["ID"].notna().all()
    assert data["ID"].is_unique
    weight = pd.to_numeric(data["WT_DOM"], errors="coerce")
    assert weight.notna().all()
    assert weight.gt(0).all()
    raw_data_by_year[year] = data
    metadata_by_year[year] = metadata

inventory = pd.DataFrame(
    {
        "연도": YEARS,
        "행 수": [len(raw_data_by_year[year]) for year in YEARS],
        "공식 변수 수": [
            len(official_variables_by_year[year]) for year in YEARS
        ],
    }
)
display(inventory)  # noqa: F821  # 로드 결과 확인용 출력

,연도,행 수,공식 변수 수
0,2023,52111,1376
1,2024,51754,1235
2,2025,52185,1427


## 2023년 변수명 수정

2023년 A계열 문항 번호는 2024·2025년보다 대체로 하나 앞서 있으므로
`A2→A1`, `A3→A2`, ..., `A12→A11`처럼 문항 의미를 기준으로 이름을
이동한다(`NA*`, `PA*` 접두사도 동일). `A2`(여행사 상품 구매여부)는
3범주 파생에 원값이 그대로 필요하므로 이 자동 이동에서 제외하고
원래 이름을 유지한 채 다음 전처리 단계에서 처리한다. 정규식으로
표현되지 않는 이름 이동(비여행이유 `B9_n→ZQ1_n`)은 명시적으로
추가한다.

In [3]:
REFERENCE_COLUMNS_2024 = set(metadata_by_year[2024].column_names)
EXCLUDE_FROM_GENERIC_SHIFT = {"A2"}
EXPLICIT_2023_RENAMES = {
    "B9_1": "ZQ1_1",
    "B9_2": "ZQ1_2",
    "B9_3": "ZQ1_3",
}


def shift_2023_name(source_name: str) -> str:
    """2023년 원변수명을 2024·2025년 문항 번호 기준 이름으로 이동한다.

    Args:
        source_name: 2023년 SAV 원변수명.

    Returns:
        이동된 이름. 이동 대상이 아니거나 2024년에 대상 이름이 실제로
        없으면 원래 이름을 그대로 반환한다.
    """
    if source_name in EXPLICIT_2023_RENAMES:
        return EXPLICIT_2023_RENAMES[source_name]
    if source_name in EXCLUDE_FROM_GENERIC_SHIFT:
        return source_name
    match = re.match(r"^(N|P)?A(1[0-2]|[2-9])(.*)$", source_name)
    if match is None:
        return source_name
    target = (
        f"{match.group(1) or ''}A{int(match.group(2)) - 1}"
        f"{match.group(3)}"
    )
    return target if target in REFERENCE_COLUMNS_2024 else source_name


rename_map_2023 = {
    name: shift_2023_name(name)
    for name in official_variables_by_year[2023]
}
assert len(set(rename_map_2023.values())) == len(rename_map_2023)

working_data_by_year = {
    2023: raw_data_by_year[2023].rename(columns=rename_map_2023),
    2024: raw_data_by_year[2024],
    2025: raw_data_by_year[2025],
}
original_name_2023 = {
    new: old for old, new in rename_map_2023.items()
}

renamed_sample = {
    old: new for old, new in rename_map_2023.items() if old != new
}
print(f"2023년 이름 이동 변수 수: {len(renamed_sample):,}개")
display(  # noqa: F821  # 이름 이동 결과 표본 확인용 출력
    pd.DataFrame(
        sorted(renamed_sample.items())[:10],
        columns=["2023년 원변수명", "이동된 이름"],
    )
)

2023년 이름 이동 변수 수: 279개


,2023년 원변수명,이동된 이름
0,A10,A9
1,A10A_1,A9A_1
2,A10A_10,A9A_10
3,A10A_11,A9A_11
4,A10A_12,A9A_12
5,A10A_2,A9A_2
6,A10A_3,A9A_3
7,A10A_4,A9A_4
8,A10A_5,A9A_5
9,A10A_6,A9A_6


## 전처리: 최종 통합 스키마와 특수 처리 변수 목록

`variable-harmonization.md`와 사용자 대응표 CSV에서 단순 이름 이동으로
해결되지 않는 변수 그룹을 미리 정의한다. 나머지 공통 변수는 연도별
직접 복사로 처리한다.

In [4]:
TF_SELF_NAMES = {
    *(f"A2_{i}" for i in (1, 2, 3, 4, 5, 6, 7, 10, 11)),
    *(f"A4_{i}" for i in range(1, 22)),
}
A6B_COMBINE_GROUPS = {
    "A6B_1": ["A6B_1", "A6B_2", "A6B_4", "A6B_6"],
    "A6B_2": ["A6B_3", "A6B_5"],
    "A6B_3": ["A6B_7", "A6B_8"],
    "A6B_4": ["A6B_9"],
    "A6B_5": ["A6B_10"],
    "A6B_6": ["A6B_11"],
    "A6B_7": ["A6B_12"],
}
A6B_SOURCE_ONLY_NAMES = {"A6B_8", "A6B_9", "A6B_10", "A6B_11", "A6B_12"}
A5A_NAMES = {"A5A_1", "A5A_2", "A5A_3"}
A5A_REMAP_2024_2025 = {
    1.0: 1.0, 2.0: 1.0, 3.0: 1.0, 4.0: 2.0, 5.0: 3.0, 6.0: 4.0,
    7.0: 9.0, 8.0: 6.0, 9.0: 7.0, 10.0: 8.0, 11.0: 9.0,
}
ZQ1_NAMES = {"ZQ1_1", "ZQ1_2", "ZQ1_3"}
B7_TARGET_PATTERN = re.compile(r"^D_TRA([1-6])_B7_([1-3])$")
B7A_SOURCE_PATTERN = re.compile(r"^D_TRA([1-6])_B7A_([1-3])$")
B7B_SOURCE_PATTERN = re.compile(r"^D_TRA([1-6])_B7B_([1-3])$")

final_columns = ["YEAR"]
for name in official_variables_by_year[REFERENCE_YEAR]:
    if B7B_SOURCE_PATTERN.match(name):
        continue
    if name in A6B_SOURCE_ONLY_NAMES:
        continue
    match_b7a = B7A_SOURCE_PATTERN.match(name)
    if match_b7a:
        slot, rank = match_b7a.groups()
        final_columns.append(f"D_TRA{slot}_B7_{rank}")
        continue
    final_columns.append(name)

assert len(final_columns) == len(set(final_columns))
assert final_columns[:2] == ["YEAR", "ID"]
print(f"통합 스키마 열 수: {len(final_columns):,}개")

통합 스키마 열 수: 1,405개


## 전처리: 여행사 패키지 구매(A1) 파생

2023년은 구매여부(`A2`: 1=예/2=아니오)와 패키지 종류(`A2B`: 1=전체/
2=부분)를 2024·2025년 `A1`의 3범주(1=전체 패키지, 2=부분 패키지,
3=구매 안함)로 합친다. 2024·2025년은 원값을 그대로 사용한다.

In [5]:
def build_a1(year: int, source_data: pd.DataFrame) -> pd.Series:
    """여행사 패키지 구매 여부·종류를 3범주 통합 코드로 만든다.

    Args:
        year: 조사 연도.
        source_data: 해당 연도의 작업용 데이터프레임.

    Returns:
        통합 코드(1=전체 패키지, 2=부분 패키지, 3=구매 안함) 시리즈.
    """
    if year != 2023:
        return source_data["A1"].astype("float64")
    purchased = source_data["A2"]
    package_type = source_data["A2B"]
    result = pd.Series(np.nan, index=source_data.index, dtype="float64")
    result.loc[purchased.eq(2.0)] = 3.0
    result.loc[purchased.eq(1.0)] = package_type.loc[purchased.eq(1.0)]
    return result


a1_by_year = {
    year: build_a1(year, working_data_by_year[year]) for year in YEARS
}
for year in YEARS:
    print(
        f"{year}년 A1 분포:",
        a1_by_year[year].value_counts(dropna=False).sort_index().to_dict(),
    )

2023년 A1 분포: {1.0: 372, 2.0: 86, 3.0: 23824, nan: 27829}
2024년 A1 분포: {1.0: 416, 2.0: 156, 3.0: 23291, nan: 27891}
2025년 A1 분포: {1.0: 450, 2.0: 116, 3.0: 23422, nan: 28197}


## 전처리: 이동수단(B7) 파생 — 2023년

`D_TRA{k}_CHECK == "Y"`인 A계열 응답 대상 여행 슬롯에 `A1_1~3`(2023년
원 이동수단) 값을 `D_TRA{k}_B7_1~3`으로 옮긴다. 2023년 코드 11(기타)은
2024·2025년 B7A/B7B 코드 체계의 12(기타)로 재코딩한 뒤 할당한다.
CASE가 있지만 CHECK가 "Y"가 아닌 실제 여행은 이동수단 미조사로 보고
구조적 결측 코드 -1을 넣는다. CASE 자체가 없는 슬롯(여행 없음)은
결측으로 둔다.

In [6]:
def build_b7_2023(source_data: pd.DataFrame) -> dict[str, pd.Series]:
    """2023년 CHECK 대상 여행 슬롯에 이동수단 순위를 파생한다.

    Args:
        source_data: 2023년 작업용 데이터프레임(이름 이동 적용 후).

    Returns:
        `D_TRA{slot}_B7_{rank}` 이름의 파생 시리즈 사전.
    """
    check_columns = [f"D_TRA{slot}_CHECK" for slot in range(1, 7)]
    check_y_count = source_data[check_columns].eq("Y").sum(axis=1)
    assert check_y_count.le(1).all()

    result = {}
    for slot in range(1, 7):
        case_exists = source_data[f"D_TRA{slot}_CASE"].notna()
        selected = source_data[f"D_TRA{slot}_CHECK"].eq("Y")
        assert (~selected | case_exists).all()
        for rank in (1, 2, 3):
            recoded_source = source_data[f"A1_{rank}"].replace(
                {11.0: 12.0}
            )
            derived = pd.Series(
                np.nan, index=source_data.index, dtype="float64"
            )
            derived.loc[case_exists & ~selected] = -1.0
            derived.loc[selected] = recoded_source.loc[selected]
            result[f"D_TRA{slot}_B7_{rank}"] = derived
    return result


b7_2023 = build_b7_2023(working_data_by_year[2023])
print(f"2023년 B7 파생 열 수: {len(b7_2023):,}개")

2023년 B7 파생 열 수: 18개


## 전처리: 이동수단(B7) 파생 — 2024·2025년

여행 슬롯별 지역간(`B7A_1~3`)과 지역내(`B7B_1~3`) 이동수단을 지역간→
지역내 순서로 이어 붙인 뒤 중복 코드를 제거하고 앞에서부터 3개를
1~3순위로 재계산한다.

In [7]:
def top_three_unique(values: list[float]) -> list[float]:
    """결측이 아닌 값 중 처음 등장한 순서로 최대 3개를 고른다.

    Args:
        values: 지역간·지역내 이동수단 코드를 순서대로 나열한 값.

    Returns:
        중복을 제거한 상위 3개 값. 부족하면 NaN으로 채운다.
    """
    picked: list[float] = []
    for value in values:
        if pd.notna(value) and value not in picked:
            picked.append(value)
        if len(picked) == 3:
            break
    return picked + [np.nan] * (3 - len(picked))


def build_b7_2024_2025(
    year: int, source_data: pd.DataFrame
) -> dict[str, pd.Series]:
    """지역간·지역내 이동수단을 합쳐 1~3순위를 다시 계산한다.

    Args:
        year: 조사 연도(2024 또는 2025).
        source_data: 해당 연도의 작업용 데이터프레임.

    Returns:
        `D_TRA{slot}_B7_{rank}` 이름의 파생 시리즈 사전.
    """
    slots = range(1, 6) if year == 2024 else range(1, 7)
    result = {}
    for slot in slots:
        columns = [f"D_TRA{slot}_B7A_{rank}" for rank in (1, 2, 3)] + [
            f"D_TRA{slot}_B7B_{rank}" for rank in (1, 2, 3)
        ]
        combined = source_data[columns].apply(
            lambda row: top_three_unique(list(row)),
            axis=1,
            result_type="expand",
        )
        for index, rank in enumerate((1, 2, 3)):
            result[f"D_TRA{slot}_B7_{rank}"] = combined[index]
    return result


b7_by_year = {2023: b7_2023}
for year in (2024, 2025):
    b7_by_year[year] = build_b7_2024_2025(year, working_data_by_year[year])
print(
    "2024·2025년 B7 파생 열 수:",
    {year: len(b7_by_year[year]) for year in (2024, 2025)},
)

2024·2025년 B7 파생 열 수: {2024: 15, 2025: 18}


## 전처리: 사전예약사항·여행지 활동 T/F 전환

사전예약사항(`A2_1~7,10,11`)과 여행지에서의 활동(`A4_1~21`)은 항목별
슬롯에 값이 있으면 그 항목을 선택한 것이다(값 자체가 항목 번호와
같다). 항목별 선택 여부를 0/1로 전환한다. 두 연도 모두 이름 이동
이후 동일한 이름을 쓰므로 같은 함수로 처리한다.

In [8]:
def build_tf_self(name: str, source_data: pd.DataFrame) -> pd.Series:
    """같은 이름의 원항목 값이 있으면 1, 없으면 0으로 바꾼다.

    Args:
        name: 통합 열 이름(원자료의 항목 열 이름과 같다).
        source_data: 해당 연도의 작업용 데이터프레임.

    Returns:
        0.0/1.0 값의 시리즈. 원자료에 항목이 없는 연도는 전부 0.0이다.
    """
    if name not in source_data.columns:
        return pd.Series(0.0, index=source_data.index, dtype="float64")
    return source_data[name].notna().astype("float64")


tf_self_preview = {
    year: build_tf_self("A2_1", working_data_by_year[year]).mean()
    for year in YEARS
}
print("A2_1(사전예약: 숙박시설) 선택 비율 표본:", tf_self_preview)

A2_1(사전예약: 숙박시설) 선택 비율 표본: {2023: np.float64(0.124196426858053), 2024: np.float64(0.12257989720601306), 2025: np.float64(0.13043978154642139)}


## 전처리: 동반자 유형(A6B_1~7) 결합과 T/F 전환

2023년은 이름 이동 후 자신의 7개 슬롯을 그대로 T/F 전환한다.
2024·2025년은 12범주를 2023년 7범주 그룹으로 묶어, 그룹에 속한 원항목
중 하나라도 값이 있으면 1로 표시한다.

In [9]:
def build_a6b(name: str, year: int, source_data: pd.DataFrame) -> pd.Series:
    """동반자 유형을 2023년 7범주 기준 T/F로 통합한다.

    Args:
        name: 통합 열 이름(`A6B_1`~`A6B_7`).
        year: 조사 연도.
        source_data: 해당 연도의 작업용 데이터프레임.

    Returns:
        0.0/1.0 값의 시리즈.
    """
    if year == 2023:
        return build_tf_self(name, source_data)
    sources = A6B_COMBINE_GROUPS[name]
    return source_data[sources].notna().any(axis=1).astype("float64")


a6b_preview = {
    year: build_a6b("A6B_1", year, working_data_by_year[year]).mean()
    for year in YEARS
}
print("A6B_1(가족 동반) 선택 비율 표본:", a6b_preview)

A6B_1(가족 동반) 선택 비율 표본: {2023: np.float64(0.2580069467099077), 2024: np.float64(0.2597287166209375), 2025: np.float64(0.25859921433362076)}


## 전처리: 참고 인터넷 사이트(A5A_1~3) 코드 재매핑

2024·2025년 11범주를 2023년 9범주 체계로 재매핑한다(1+2+3→1, 4→2,
5→3, 6→4, 7→9, 8→6, 9→7, 10→8, 11→9). 2023년 자체 값 5(여행 관련
블로그)는 순위와 무관하게 9(기타)로 통일한다(사용자 확인 완료).

In [10]:
def build_a5a(name: str, year: int, source_data: pd.DataFrame) -> pd.Series:
    """참고한 인터넷 사이트 코드를 2023년 9범주 체계로 재매핑한다.

    Args:
        name: 통합 열 이름(`A5A_1`~`A5A_3`).
        year: 조사 연도.
        source_data: 해당 연도의 작업용 데이터프레임.

    Returns:
        재매핑된 코드 시리즈.
    """
    if name not in source_data.columns:
        return pd.Series(np.nan, index=source_data.index, dtype="float64")
    if year == 2023:
        return source_data[name].replace({5.0: 9.0})
    return source_data[name].map(A5A_REMAP_2024_2025)


a5a_preview = {
    year: build_a5a("A5A_1", year, working_data_by_year[year])
    .value_counts(dropna=False)
    .sort_index()
    .to_dict()
    for year in YEARS
}
for year in YEARS:
    print(f"{year}년 A5A_1 재매핑 분포:", a5a_preview[year])

2023년 A5A_1 재매핑 분포: {1.0: 3949, 2.0: 162, 3.0: 766, 4.0: 647, 6.0: 30, 7.0: 69, 8.0: 86, 9.0: 666, nan: 45736}
2024년 A5A_1 재매핑 분포: {1.0: 3562, 2.0: 251, 3.0: 874, 4.0: 822, 6.0: 52, 7.0: 93, 8.0: 41, 9.0: 23, nan: 46036}
2025년 A5A_1 재매핑 분포: {1.0: 3764, 2.0: 317, 3.0: 845, 4.0: 758, 6.0: 48, 7.0: 81, 8.0: 38, 9.0: 74, nan: 46260}


## 전처리: 비여행이유(ZQ1_1~3) 코드 통합

2024·2025년 코드 11(최근에 여행을 다녀와서)과 12(기타)는 2023년 코드
11(기타)로 합친다. 2023년 자체 코드 21(코로나19)도 11(기타)로 합친다.
2023년 이름은 앞서 `ZQ1_1~3`으로 이동해 두었다.

In [11]:
def build_zq1(name: str, year: int, source_data: pd.DataFrame) -> pd.Series:
    """비여행이유 코드를 2023년 11=기타 체계로 통합한다.

    Args:
        name: 통합 열 이름(`ZQ1_1`~`ZQ1_3`).
        year: 조사 연도.
        source_data: 해당 연도의 작업용 데이터프레임.

    Returns:
        통합 코드 시리즈.
    """
    if name not in source_data.columns:
        return pd.Series(np.nan, index=source_data.index, dtype="float64")
    if year == 2023:
        return source_data[name].replace({21.0: 11.0})
    return source_data[name].replace({12.0: 11.0})


zq1_preview = {
    year: build_zq1("ZQ1_1", year, working_data_by_year[year])
    .value_counts(dropna=False)
    .sort_index()
    .to_dict()
    for year in YEARS
}
for year in YEARS:
    print(f"{year}년 ZQ1_1 통합 분포:", zq1_preview[year])

2023년 ZQ1_1 통합 분포: {1.0: 1696, 2.0: 589, 3.0: 2034, 4.0: 10141, 5.0: 1556, 6.0: 724, 7.0: 327, 8.0: 1434, 9.0: 6762, 10.0: 579, 11.0: 181, nan: 26088}
2024년 ZQ1_1 통합 분포: {1.0: 2065, 2.0: 637, 3.0: 2111, 4.0: 9813, 5.0: 1248, 6.0: 647, 7.0: 285, 8.0: 1251, 9.0: 4546, 10.0: 512, 11.0: 2582, nan: 26057}
2025년 ZQ1_1 통합 분포: {1.0: 2397, 2.0: 608, 3.0: 2051, 4.0: 9611, 5.0: 1028, 6.0: 647, 7.0: 291, 8.0: 1251, 9.0: 4295, 10.0: 505, 11.0: 3210, nan: 26291}


## 전처리: 최종 통합 스키마 정의와 3개년 결합

연도별 작업용 데이터프레임에 위에서 만든 파생값을 반영해 통합 스키마
순서로 정렬한 뒤 세로로 이어 붙인다. 직접 복사 대상이지만 해당 연도에
원변수가 없으면(예: 2024년 `D_TRA6_*`) 결측으로 둔다.

In [12]:
def assemble_year(year: int) -> pd.DataFrame:
    """한 연도의 작업용 데이터프레임을 통합 스키마로 변환한다.

    열을 하나씩 대입하면 데이터프레임이 조각나 성능 경고가 반복되므로,
    시리즈를 사전에 모았다가 한 번에 합친다.

    Args:
        year: 조사 연도.

    Returns:
        `final_columns` 순서의 데이터프레임.
    """
    source_data = working_data_by_year[year]
    columns: dict[str, pd.Series] = {
        "YEAR": pd.Series(year, index=source_data.index, dtype="int64")
    }

    for name in final_columns[1:]:
        if name == "A1":
            columns[name] = a1_by_year[year]
        elif name in TF_SELF_NAMES:
            columns[name] = build_tf_self(name, source_data)
        elif name in A6B_COMBINE_GROUPS:
            columns[name] = build_a6b(name, year, source_data)
        elif name in A5A_NAMES:
            columns[name] = build_a5a(name, year, source_data)
        elif name in ZQ1_NAMES:
            columns[name] = build_zq1(name, year, source_data)
        elif B7_TARGET_PATTERN.match(name):
            columns[name] = b7_by_year[year].get(
                name, pd.Series(np.nan, index=source_data.index)
            )
        elif name in source_data.columns:
            columns[name] = source_data[name]
        else:
            columns[name] = pd.Series(
                np.nan, index=source_data.index, dtype="float64"
            )

    return pd.concat(columns, axis=1).reindex(columns=final_columns)


integrated_by_year = {year: assemble_year(year) for year in YEARS}
integrated = pd.concat(
    [integrated_by_year[year] for year in YEARS],
    ignore_index=True,
)

assert integrated.columns.tolist() == final_columns
assert len(integrated) == sum(EXPECTED_ROWS.values())
assert integrated[["YEAR", "ID"]].drop_duplicates().shape[0] == len(
    integrated
)
print(f"통합 데이터 크기: {integrated.shape[0]:,}행 × {integrated.shape[1]:,}열")

통합 데이터 크기: 156,050행 × 1,405열


## 데이터 미리보기

In [13]:
preview_columns = [
    "YEAR", "ID", "WT_DOM", "A1", "A2_1", "A4_1", "A4A", "A5A_1",
    "A6B_1", "ZQ1_1", "D_TRA1_B7_1", "D_TRA1_CASE", "D_TRA1_CHECK",
]
display(integrated[preview_columns].head(5))  # noqa: F821  # 통합 결과 표본 확인용 출력
display(  # noqa: F821  # 연도별 행 수 확인용 출력
    integrated.groupby("YEAR").size().rename("행 수").reset_index()
)

,YEAR,ID,WT_DOM,A1,A2_1,A4_1,A4A,A5A_1,A6B_1,ZQ1_1,D_TRA1_B7_1,D_TRA1_CASE,D_TRA1_CHECK
0,2023,11010560931_124820,26559.502481,NaN,0.0,0.0,NaN,NaN,0.0,10.0,NaN,NaN,
1,2023,11010560931_124821,33746.017174,3.0,0.0,0.0,18.0,NaN,1.0,NaN,1.0,1.0,Y
2,2023,11010560931_124823,15625.084605,NaN,0.0,0.0,NaN,NaN,0.0,3.0,NaN,NaN,
3,2023,11010560931_124825,15620.054419,NaN,0.0,0.0,NaN,NaN,0.0,3.0,NaN,NaN,
4,2023,11010560931_124833,7209.363279,3.0,0.0,1.0,1.0,NaN,0.0,NaN,1.0,1.0,Y


,YEAR,행 수
0,2023,52111
1,2024,51754
2,2025,52185


## 코드북 작성

통합 데이터의 각 열마다 연도별 원천 정보와 처리 방식을 기록한 변수
사전을 만든다.

In [14]:
def json_labels(labels: dict) -> str:
    """값 라벨 사전을 안정적인 JSON 문자열로 바꾼다.

    Args:
        labels: 코드와 라벨의 대응 사전.

    Returns:
        코드 키를 문자열로 만든 JSON. 라벨이 없으면 빈 문자열.
    """
    if not labels:
        return ""
    normalized = {
        str(int(key) if float(key).is_integer() else key): str(value)
        for key, value in labels.items()
    }
    return json.dumps(normalized, ensure_ascii=False, sort_keys=True)


TF_VALUE_LABELS = json_labels({0: "미해당", 1: "해당"})


def source_name_for(year: int, name: str) -> str:
    """통합 열 이름에 대응하는 연도별 원변수명을 찾는다.

    Args:
        year: 조사 연도.
        name: 통합 열 이름.

    Returns:
        원변수명. 없으면 빈 문자열.
    """
    if year == 2023:
        if name == "A1":
            return "A2, A2B"
        if name in A5A_NAMES or name in ZQ1_NAMES:
            return original_name_2023.get(name, "")
        if name in A6B_COMBINE_GROUPS:
            return original_name_2023.get(name, "")
        if B7_TARGET_PATTERN.match(name):
            return "A1_1, A1_2, A1_3"
        return original_name_2023.get(name, name)
    if name in A6B_COMBINE_GROUPS and year != 2023:
        return ", ".join(A6B_COMBINE_GROUPS[name])
    match_b7 = B7_TARGET_PATTERN.match(name)
    if match_b7 and year != 2023:
        slot = match_b7.group(1)
        rank = match_b7.group(2)
        return f"D_TRA{slot}_B7A_{rank}, D_TRA{slot}_B7B_{rank}"
    return name if name in working_data_by_year[year].columns else ""

def get_column_type(name: str) -> str:
    """통합 변수의 SPSS 측정 수준을 반환한다.

    Args:
        name: 통합 데이터의 변수명.

    Returns:
        nominal, ordinal, scale 중 하나. 확인할 수 없으면 빈 문자열.
    """
    if name == "YEAR":
        return "nominal"

    if name in TF_SELF_NAMES or name in A6B_COMBINE_GROUPS:
        return "nominal"

    if B7_TARGET_PATTERN.match(name):
        return "nominal"

    # 기준 연도인 2025년에서 먼저 조회
    measure = metadata_by_year[REFERENCE_YEAR].variable_measure.get(name)
    if measure:
        return measure

    # 2025년에 없거나 변수명이 변경된 경우 연도별 원변수명으로 조회
    for year in YEARS:
        source_name = source_name_for(year, name)

        # 여러 원변수를 결합한 경우 첫 번째 변수의 측정 수준 사용
        source_names = [
            source.strip()
            for source in source_name.split(",")
            if source.strip()
        ]

        for source in source_names:
            measure = metadata_by_year[year].variable_measure.get(source)
            if measure:
                return measure

    return ""

In [15]:
codebook_rows = []
for order, name in enumerate(final_columns, start=1):
    base_label = metadata_by_year[REFERENCE_YEAR].column_names_to_labels.get(
        name, ""
    )
    row = {
        "column_order": order,
        "column_name": name,
        "column_label": base_label,
        "column_type": get_column_type(name),
        "comparison_status": "direct",
        "integrated_value_labels": "",
        "structural_missing_code": "",
        "notes": "",
    }

    if name == "YEAR":
        row["column_label"] = "조사 연도"
        row["comparison_status"] = "derived_with_assumption"
        row["integrated_value_labels"] = json.dumps(
            {str(year): f"{year}년" for year in YEARS}, ensure_ascii=False
        )
    elif name == "A1":
        row["column_label"] = "여행사 패키지 상품 구매 유형(3범주 통합)"
        row["comparison_status"] = "harmonized"
        row["integrated_value_labels"] = json_labels(
            metadata_by_year[REFERENCE_YEAR].variable_value_labels.get(
                "A1", {}
            )
        )
        row["notes"] = (
            "2023년은 A2(구매여부)·A2B(패키지 종류)를 조합, "
            "2024·2025년은 원값을 그대로 사용"
        )
    elif name in TF_SELF_NAMES:
        slot_match = re.search(r"_(\d+)$", name)
        slot_labels = metadata_by_year[REFERENCE_YEAR].variable_value_labels.get(
            name, {}
        )
        item_label = slot_labels.get(float(slot_match.group(1)))
        if item_label:
            row["column_label"] = re.sub(
                r"-\d+$", f"_{item_label}", base_label
            )
        row["comparison_status"] = "harmonized"
        row["integrated_value_labels"] = TF_VALUE_LABELS
        row["notes"] = "항목 슬롯 값의 존재 여부를 0/1로 전환"
    elif name in A6B_COMBINE_GROUPS:
        original_label = metadata_by_year[2023].column_names_to_labels.get(
            original_name_2023.get(name, ""), ""
        )
        row["column_label"] = f"동반자 유형: {original_label}" if original_label else base_label
        row["comparison_status"] = "harmonized"
        row["integrated_value_labels"] = TF_VALUE_LABELS
        row["notes"] = (
            "2023년은 자기 슬롯 T/F, 2024·2025년은 "
            f"{A6B_COMBINE_GROUPS[name]} 항목 중 하나라도 있으면 1"
        )
    elif name in A5A_NAMES:
        row["comparison_status"] = "harmonized"
        row["integrated_value_labels"] = json_labels(
            metadata_by_year[2023].variable_value_labels.get(
                original_name_2023.get(name, ""), {}
            )
        )
        row["notes"] = (
            "값 체계는 2023년 9범주 기준. 2024·2025년 11범주는 "
            "1+2+3→1,4→2,5→3,6→4,7→9,8→6,9→7,10→8,11→9로 재매핑; "
            "2023년 자체 값 5(블로그)는 9(기타)로 재코딩"
        )
    elif name in ZQ1_NAMES:
        labels_2023 = dict(
            metadata_by_year[2023].variable_value_labels.get(
                original_name_2023.get(name, ""), {}
            )
        )
        labels_2023.pop(21.0, None)
        row["comparison_status"] = "harmonized"
        row["integrated_value_labels"] = json_labels(labels_2023)
        row["notes"] = (
            "값 체계는 2023년 11=기타 기준. 2024·2025년 코드 "
            "11(최근 여행)·12(기타)와 2023년 코드 21(코로나19)를 "
            "11(기타)로 통합"
        )
    elif B7_TARGET_PATTERN.match(name):
        slot, rank = B7_TARGET_PATTERN.match(name).groups()
        labels = dict(
            metadata_by_year[REFERENCE_YEAR].variable_value_labels.get(
                f"D_TRA{slot}_B7A_1", {}
            )
        )
        labels[-1.0] = "해당 여행의 이동수단 미조사"
        row["column_label"] = (
            f"{slot}차 여행 주요 이동수단(지역간·지역내 통합) {rank}순위"
        )
        row["comparison_status"] = (
            "partially_available" if slot == "6" else "derived_with_assumption"
        )
        row["integrated_value_labels"] = json_labels(labels)
        row["structural_missing_code"] = "-1=해당 여행의 이동수단 미조사"
        row["notes"] = (
            "2023년은 CHECK='Y' 슬롯에 A1_1~3(11→12 재코딩)을 배정, "
            "CASE는 있으나 CHECK가 'Y'가 아니면 -1; 2024·2025년은 "
            "B7A_1~3과 B7B_1~3을 지역간→지역내 순으로 이어붙여 "
            "중복 제거 후 상위 3개로 재계산"
        )
    else:
        row["integrated_value_labels"] = json_labels(
            metadata_by_year[REFERENCE_YEAR].variable_value_labels.get(
                name, {}
            )
        )
        if name.startswith("D_TRA6_"):
            row["comparison_status"] = "partially_available"
            row["notes"] = "2024년 D_TRA6 미수집; 2023·2025년 원값 보존"
        elif original_name_2023.get(name, name) != name:
            row["comparison_status"] = "harmonized"
            row["notes"] = "2023년 문항 번호를 2024·2025년 기준 이름으로 이동"
        if name.endswith("_CASE"):
            prefix = f"{row['notes']}; " if row["notes"] else ""
            row["notes"] = (
                f"{prefix}코드북 정의와 별도로 현재 SAV 실제 관측 범위는 1~5"
            )
        if re.match(r"^D_TRA[1-6]_CHECK$", name):
            row["notes"] = (
                "Y=A계열 본설문 응답 대상 여행; 여행 존재 여부는 "
                "CHECK가 아니라 같은 슬롯의 CASE 비결측으로 판정"
            )
        if name == "A4A":
            row["notes"] = (
                "2025년 코드 16의 의미가 확대(드라마 촬영지→공연장 및 "
                "드라마/영화 촬영지); 연도 간 카테고리 비교 시 유의"
            )

    for year in YEARS:
        source_name = source_name_for(year, name)
        available = bool(source_name) or name == "YEAR"
        row[f"y{year}_source_name"] = source_name
        row[f"y{year}_available"] = available

    codebook_rows.append(row)

codebook = pd.DataFrame(codebook_rows)
assert len(codebook) == len(final_columns)
assert codebook["column_name"].is_unique
print(f"코드북 크기: {codebook.shape[0]:,}행 × {codebook.shape[1]:,}열")
display(  # noqa: F821  # 코드북 표본 확인용 출력
    codebook.loc[
        codebook["column_name"].isin(
            ["YEAR", "A1", "A2_1", "A5A_1", "A6B_1", "ZQ1_1", "D_TRA1_B7_1"]
        ),
        ["column_name", "column_type", "comparison_status", "notes"],
    ]
)

코드북 크기: 1,405행 × 14열


,column_name,column_type,comparison_status,notes
0,YEAR,nominal,derived_with_assumption,
25,D_TRA1_B7_1,nominal,derived_with_assumption,"2023년은 CHECK='Y' 슬롯에 A1_1~3(11→12 재코딩)을 배정, CA..."
1026,A1,nominal,harmonized,"2023년은 A2(구매여부)·A2B(패키지 종류)를 조합, 2024·2025년은 원..."
1028,A2_1,nominal,harmonized,항목 슬롯 값의 존재 여부를 0/1로 전환
1069,A5A_1,nominal,harmonized,"값 체계는 2023년 9범주 기준. 2024·2025년 11범주는 1+2+3→1,4..."
1075,A6B_1,nominal,harmonized,"2023년은 자기 슬롯 T/F, 2024·2025년은 ['A6B_1', 'A6B_2..."
1340,ZQ1_1,nominal,harmonized,값 체계는 2023년 11=기타 기준. 2024·2025년 코드 11(최근 여행)·...


## 통합 데이터와 코드북 저장

`data/preprocess/`에 기존 `_old` 파일명에서 `_old`를 제거한 이름으로
저장한다(UTF-8-SIG, CSV).

In [19]:
PREPROCESS_DATA_DIR.mkdir(parents=True, exist_ok=True)
integrated.to_csv(OUTPUT_DATA_PATH, index=False, encoding="utf-8-sig")
codebook.to_csv(OUTPUT_CODEBOOK_PATH, index=False, encoding="utf-8-sig")
print(f"통합 데이터 저장: {OUTPUT_DATA_PATH}")
print(f"코드북 저장: {OUTPUT_CODEBOOK_PATH}")

통합 데이터 저장: C:\Users\ilove\Desktop\tourism_poster\data\preprocess\national_travel_survey\national_travel_survey_2023_2025.csv
코드북 저장: C:\Users\ilove\Desktop\tourism_poster\data\preprocess\national_travel_survey\national_travel_survey_2023_2025_codebook.csv


## 핵심 결과와 한계

- 2023·2024·2025년 국내여행 SAV를 `YEAR + ID` 기준 wide 형식으로
  통합했다. 사전예약사항·여행지 활동·동반자 유형은 항목별 T/F로,
  여행사 패키지 구매·참고 인터넷 사이트·비여행이유·이동수단은 값
  체계를 통일한 재코딩으로 통합했다.
- 여행 존재 여부는 `D_TRA*_CASE` 비결측으로 판정하며 `D_TRA*_CHECK`는
  A계열 본설문 응답 대상 여행 식별에만 사용한다.
- `WT_DOM`은 연도별 모집단 추정용 가중치이며 세 연도 합을 고유 인구로
  해석하지 않는다.
- 값 라벨을 확인하지 않고 숫자 코드를 의미로 단정하지 않는다. 코드북의
  `integrated_value_labels`와 `notes`를 함께 확인한다.
- 참고 인터넷 사이트(A5A_1~3)의 2023년 값 5(블로그)를 9(기타)로 통일한
  결정과 비여행이유 통합 변수명(ZQ1_1~3)은 사용자 확인을 거쳤다.